#### In Tokenization method NLP, Can I assume padding always assignee by 0 & CREATING ATTENTION MASKS always 1. And 0 & 1 reserved for padding and CREATING ATTENTION MASKS.

No, you should not assume that padding always uses token ID 0 or that 0 and 1 are reserved exclusively for padding and attention masks. These are two different things in NLP tokenization.

Does every NLP tokenizer use padding ID 0?

No. The padding ID depends on the tokenizer and model vocabulary


| Model/tokenizer | Padding token | Padding ID |
| :--- | :--- | :--- |
| DistilBERT (`distilbert-base-uncased`) | `[PAD]` | 0 |
| BERT (`bert-base-uncased`) | `[PAD]` | 0 |
| GPT-2 (original tokenizer configuration) | No dedicated padding token by default | Not defined until configured |
| Other models | Model-dependent | Model-dependent |

Also, padding is not always added. If sequences already have equal lengths, or you do not request padding, the tokenizer may not add any padding tokens.

Padding IDs are defined by the tokenizer's vocabulary, while attention-mask values describe which sequence positions are valid. In many Hugging Face Transformer models, 1 means a valid token and 0 means padding, but you should verify the model's conventions rather than treating these values as universal.

#### What is Contextual Embeddings (BERT, GPT, Transformer models) & how it is different from Word Embeddings?

### Contextual Embeddings vs. Word Embeddings

The main difference is this:

Traditional word embeddings give a word a relatively fixed vector, whereas contextual embeddings generate a representation that depends on how the word is used in a sentence.

You have already worked with TF-IDF and DistilBERT. This is the next important step in understanding how modern NLP models represent language.

#### 1. Traditional word embeddings

Traditional word embeddings represent each word as a dense numerical vector. Models such as Word2Vec and GloVe learn these vectors from patterns of word co-occurrence in text.

For example, consider these sentences:

* "I deposited money at the **bank**."
* "The **bank** of the river was flooded."

A traditional Word2Vec model typically assigns the same vector to **bank** in both sentences.

#### 2. Contextual embeddings

Contextual embeddings are representations generated by a neural language model using the surrounding tokens as context.

Models such as BERT, DistilBERT, and GPT use Transformer architectures to process sequences. Through attention and other neural-network operations, they build representations that incorporate information from surrounding words.

Using the same example:

* "I deposited money at the **bank**."
* "The **bank** of the river was flooded."

A contextual model can generate different vectors for the two occurrences of **bank**.
#### 3. What is the actual difference?

| Feature | Traditional word embeddings | Contextual embeddings |
| :--- | :--- | :--- |
| Examples | Word2Vec, GloVe | BERT, DistilBERT, GPT hidden states |
| Vector for a word | Usually fixed for each vocabulary word | Depends on context |
| Understands multiple word senses | Limited | Better able to distinguish senses |
| Uses surrounding words | During training, to learn word associations | During representation generation, through the model |
| Architecture | Often shallow neural networks or statistical methods | Transformer neural networks |
| Computational cost | Generally low | Generally higher |
| Example use | Word similarity, lightweight NLP | Semantic search, contextual classification, modern NLP |

#### How does the Transformer create contextual representations?

***Q. In transformer architectures, the input embedding layer converts tokens into dense vectors. Word embeddings map each token to a fixed‑dimensional vector (e.g., sequence length × hidden size, such as 50 × 768). Since transformers process tokens in parallel, they lack inherent sequence order. To encode positional information, sinusoidal functions (sine and cosine at varying frequencies) are added to the embeddings. This positional encoding allows the model to capture word order and relational structure within the sequence.***

- Input embeddings convert token IDs into vectors; they do not necessarily represent complete word meanings yet.
- Positional encoding does not calculate relationships between words using cosine. It provides information about token 
positions. The Transformer learns relationships through its attention mechanism.

![Transformer create contextual representations](image-26.png)

In [1]:

import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForMaskedLM

# --------------------------------------------------
# Step 1: Load the pretrained tokenizer and model
# --------------------------------------------------
checkpoint = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(checkpoint)

model = AutoModelForMaskedLM.from_pretrained(
    checkpoint,
    output_hidden_states=True
)

model.eval()

print("Model loaded successfully.")
print("Padding token:", tokenizer.pad_token)
print("Padding token ID:", tokenizer.pad_token_id)

# --------------------------------------------------
# Step 2: Define the input sentence
# --------------------------------------------------
sentence = "The bank of the river was flooded"

# --------------------------------------------------
# Step 3: Tokenize the sentence
# --------------------------------------------------
inputs = tokenizer(
    sentence,
    return_tensors="pt",
    padding=True,
    truncation=True,
    return_attention_mask=True
)

tokens = tokenizer.convert_ids_to_tokens(
    inputs["input_ids"][0]
)

print("\nOriginal Sentence:", sentence)
print("Tokenized List:", tokens)
print("Input IDs:", inputs["input_ids"])
print("Attention Mask:", inputs["attention_mask"])

# --------------------------------------------------
# Step 4: Generate contextual embeddings
# --------------------------------------------------
with torch.no_grad():
    outputs = model(**inputs)

# Last Transformer layer's hidden states
embeddings = outputs.hidden_states[-1]

# --------------------------------------------------
# Step 5: Display the embedding tensor shape
# --------------------------------------------------
print("\nEmbedding Tensor Shape:", embeddings.shape)

# Expected shape: [batch_size, sequence_length, 768]

# --------------------------------------------------
# Step 6: Display the first three features per token
# --------------------------------------------------
print("\nContextual Embeddings (first 3 features):")

for tok, emb in zip(tokens, embeddings[0]):
    first_three = emb[:3].cpu().numpy()
    print(f"{tok:12s}: {np.round(first_three, 4)}")

# --------------------------------------------------
# Step 7: Extract the contextual embedding for "bank"
# --------------------------------------------------
bank_index = tokens.index("bank")

bank_embedding = embeddings[0, bank_index, :]

print("\nToken:", tokens[bank_index])
print("Bank Embedding Shape:", bank_embedding.shape)
print("First 10 Features:", bank_embedding[:10].cpu().numpy())


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Model loaded successfully.
Padding token: [PAD]
Padding token ID: 0

Original Sentence: The bank of the river was flooded
Tokenized List: ['[CLS]', 'the', 'bank', 'of', 'the', 'river', 'was', 'flooded', '[SEP]']
Input IDs: tensor([[  101,  1996,  2924,  1997,  1996,  2314,  2001, 10361,   102]])
Attention Mask: tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1]])

Embedding Tensor Shape: torch.Size([1, 9, 768])

Contextual Embeddings (first 3 features):
[CLS]       : [-0.2525 -0.1763 -0.0253]
the         : [-0.2555 -0.5252 -0.3493]
bank        : [-0.3196 -0.1371 -0.1627]
of          : [-0.6903 -0.2284 -0.2304]
the         : [-0.2984 -0.2301 -0.2826]
river       : [ 0.1489  0.0655 -0.3806]
was         : [-0.3888 -0.6126 -0.054 ]
flooded     : [ 0.0669 -0.3082 -0.0437]
[SEP]       : [ 0.6069  0.2829 -0.2816]

Token: bank
Bank Embedding Shape: torch.Size([768])
First 10 Features: [-0.3196481  -0.1371336  -0.16265334  0.03589509  0.35063115  0.47901344
 -0.11147881  0.9461215   0.33493158 -0.20987004]


#### Understand embedding shape Matrix

Step-by-step Transformer input architecture

    ```
    ┌──────────────────────────────────────────────────────────────┐
    │  Input sentence                                              │
    │  "The bank of the river was flooded"                         │
    └──────────────────────────────┬───────────────────────────────┘
                                   │
                                   ▼
    ┌──────────────────────────────────────────────────────────────┐
    │  Tokenization                                                │
    │  Token IDs: [101, ..., 102]                                  │
    └──────────────────────────────┬───────────────────────────────┘
                                   │
                                   ▼
    ┌──────────────────────────────────────────────────────────────┐
    │  Token embedding lookup                                      │
    │  Each token ID maps to a learned vector of 768 features.     │
    └──────────────────────────────┬───────────────────────────────┘
                                   │
                                   +
    ┌──────────────────────────────────────────────────────────────┐
    │  Positional information                                      │
    │  Adds information about each token's position.               │
    └──────────────────────────────┬───────────────────────────────┘
                                   │
                                   ▼
    ┌──────────────────────────────────────────────────────────────┐
    │  Transformer encoder layers                                  │
    │  Self-attention learns relationships between tokens.         │
    └──────────────────────────────┬───────────────────────────────┘
                                   │
                                   ▼
    ┌──────────────────────────────────────────────────────────────┐
    │  Contextual embeddings                                       │
    │  Updated representations incorporating context.              │
    └──────────────────────────────────────────────────────────────┘
    ```
#### 2. Input embeddings: Is the matrix dimension \(50 \times 768\)?

You are correct that input embeddings produce vectors, but the dimensions depend on the vocabulary size and the input sequence length.

For `distilbert-base-uncased`:

* **Vocabulary size:** approximately 30,522 tokens.
* **Embedding dimension:** 768.
* **Example input sequence length:** 9 tokens.

The learned token embedding table has dimensions:

$E \in \mathbb{R}^{30522 \times 768}$

This means there are approximately 30,522 possible token vectors, each containing 768 features.

When you tokenize a sentence into 9 tokens, the embedding lookup selects 9 rows from that table:

$X_{\text{embed}} \in \mathbb{R}^{9 \times 768}$

If your sentence had 50 tokens, the resulting matrix would be:

$X_{\text{embed}} \in \mathbb{R}^{50 \times 768}$

So, $50 \times 768$ is the representation for a sequence of 50 tokens, not the complete embedding vocabulary.

#### 3. Positional encoding: Does it use cosine mathematics to find relationships?

Not exactly. You are thinking of the sinusoidal positional encoding used in the original Transformer architecture.

That approach uses sine and cosine functions at different frequencies to represent token positions. For example:

$PE(pos, 2i) = \sin\left(\frac{pos}{10000^{2i/d}}\right)$

$PE(pos, 2i+1) = \cos\left(\frac{pos}{10000^{2i/d}}\right)$

Here:

* $pos$ is the token position.
* $i$ identifies a dimension pair.
* $d$ is the embedding dimension.

These functions generate position-dependent vectors. They do **not** calculate semantic similarity or directly determine which words are related.

Also, an important distinction for your current work: **DistilBERT uses learned positional embeddings**, not the original Transformer's fixed sinusoidal positional encoding.

In a BERT-style model, token embeddings and learned positional embeddings are combined before entering the Transformer layers.

$X_0 = X_{\text{token}} + X_{\text{position}}$

DistilBERT also applies its embedding-layer normalization and dropout after combining its embeddings.

### 2. Input embeddings: Is the matrix dimension $50 \times 768$?

You are correct that input embeddings produce vectors, but the dimensions depend on the vocabulary size and the input sequence length.

For `distilbert-base-uncased`:
* **Vocabulary size:** approximately 30,522 tokens.
* **Embedding dimension:** 768.
* **Example input sequence length:** 9 tokens.

The learned token embedding table has dimensions:

$E \in \mathbb{R}^{30522 \times 768}$

This means there are approximately 30,522 possible token vectors, each containing 768 features.

When you tokenize a sentence into 9 tokens, the embedding lookup selects 9 rows from that table:

$X_{\text{embed}} \in \mathbb{R}^{9 \times 768}$

If your sentence had 50 tokens, the resulting matrix would be:

$X_{\text{embed}} \in \mathbb{R}^{50 \times 768}$

So, $50 \times 768$ is the representation for a sequence of 50 tokens, not the complete embedding vocabulary.

### 3. Positional encoding: Does it use cosine mathematics to find relationships?

Not exactly. You are thinking of the sinusoidal positional encoding used in the original Transformer architecture.

That approach uses sine and cosine functions at different frequencies to represent token positions. For example:

$PE(\text{pos}, 2i) = \sin\left(\frac{\text{pos}}{10000^{2i/d}}\right)$

$PE(\text{pos}, 2i + 1) = \cos\left(\frac{\text{pos}}{10000^{2i/d}}\right)$

Here:
* `pos` is the token position.
* `i` identifies a dimension pair.
* `d` is the embedding dimension.

These functions generate position-dependent vectors. They do not calculate semantic similarity or directly determine which words are related.

Also, an important distinction for your current work: DistilBERT uses **learned positional embeddings**, not the original Transformer's fixed sinusoidal positional encoding.

In a BERT-style model, token embeddings and learned positional embeddings are combined before entering the Transformer layers.

$X_0 = X_{\text{token}} + X_{\text{position}}$

DistilBERT also applies its embedding-layer normalization and dropout after combining its embeddings.

#### how does the Transformer find relationships between words?

### 4. So, how does the Transformer find relationships between words?

This is the role of **self-attention**.

For each token representation $X$, the model calculates three projections:

$Q = XW_Q, \quad K = XW_K, \quad V = XW_V$

Then attention scores are calculated using:

$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$

* **Query (Q):** What information is this token looking for?
* **Key (K):** What information does each token offer for matching?
* **Value (V):** What information is passed forward?

The query-key dot products determine attention scores. These scores are normalized with softmax, and the resulting weights combine the value vectors.

#### How are attention weights optimized?

A Transformer is a neural network, and its attention matrices are optimized through the same backpropagation and optimization principles used in your other PyTorch models.

![attention weights optimized](image-27.png)

#### Weight initialization:

**Q. In this formula "$Q=XW_Q,\qquad K=XW_K,\qquad V=XW_V$" you have added weight. And As per my knowledge, weight can be initializ by glorot_uniform or he or its up to user choice. And we fins optimized weight called parameter for my neural models. Get optimized weight we use FFN and back propagations. But All that happen in Neural architecture, How this weight update here?**

Your understanding of Glorot and He initialization is correct. These are methods for assigning initial values to trainable parameters. For example, in a Transformer trained from scratch, the query matrix might be initialized using Xavier (Glorot) initialization or a model-specific initialization scheme.

This creates three trainable matrices. In a real Transformer, these are normally implemented as parameters inside an attention module, often using linear layers with biases.

Initialization methods are not restricted to Glorot or He. Transformer architectures may use normal initialization or other model-specific schemes. When you load a pretrained model, its learned weights are loaded from the checkpoint instead of being randomly initialized again.

### Step 2 — Forward propagation through attention

Once the weights are initialized, the model calculates:

$Q = XW_Q, \quad K = XW_K, \quad V = XW_V$

Next, the attention mechanism calculates:

$A = \text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)$

$Z = AV$

Here, $A$ contains the attention scores after softmax, and $Z$ contains the resulting contextual representations.

The distinction is important:
* $W_Q, W_K, W_V$ are **learnable parameters**.
* $A$ is a matrix of attention weights calculated dynamically from the current input. It is not normally a persistent parameter that the optimizer updates directly.

The optimizer learns the projection matrices, which indirectly influence the attention patterns.

### Step 3 — How does backpropagation update $W_Q, W_K, W_V$?

Suppose you train a GPT-style model to predict the next token.

The model predicts a probability distribution over possible next tokens. A loss function, such as cross-entropy, compares those predictions with the actual next token.

Backpropagation calculates the gradients:

$\frac{\partial L}{\partial W_Q}, \quad \frac{\partial L}{\partial W_K}, \quad \frac{\partial L}{\partial W_V}$

These gradients indicate how changing each parameter would affect the loss.

An optimizer then updates the parameters. For a basic gradient-descent update:

$W_Q^{\text{new}} = W_Q^{\text{old}} - \eta \frac{\partial L}{\partial W_Q}$

where $\eta$ is the learning rate.

The same principle applies to $W_K, W_V$, the feed-forward network weights, and other trainable parameters. In practice, models often use AdamW, which adapts updates using gradient statistics and may apply weight decay.

The feed-forward network does not update attention weights by itself. Rather, backpropagation propagates the loss gradients through the entire computational graph, including both attention and feed-forward layers.

### 3. How is this different when using pretrained BERT or GPT?

There are two common scenarios.

| Training scenario | What happens to the weights? |
| :--- | :--- |
| Training from scratch | Weights start from initialization and are learned through repeated optimization. |
| Using a pretrained model for inference | Learned weights are loaded from a checkpoint and remain unchanged during ordinary inference. |
| Fine-tuning a pretrained model | Existing weights are further optimized on a task-specific dataset, unless frozen. |
| LoRA fine-tuning | Typically, the original weights are frozen while additional low-rank trainable parameters are optimized. |


## Transformer 

![Step-by-step Transformer Input Architecture](image-28.png)

## Focus on Encode (Nx) Sections from transformer architecture. You have mentions Multi-Head Attentions. First tell me why its call Multi-head we need to just calculate attentions score? What is final outcome outcome from encoder(Nx)

### Transformer Encoder: Why Multi-Head Attention, and What Is Its Final Output?

Your question is exactly the right next step. We have already discussed how \(Q\), \(K\), and \(V\) are calculated. Now let's focus only on the Encoder (\(N\times\)) section of the Transformer architecture.

There are two key questions:
1. Why do we need multiple heads if one attention mechanism can calculate attention scores?
2. What exactly comes out of the encoder after its layers finish processing the input?

#### 1. Why is it called Multi-Head Attention?

A single attention head can calculate relationships between tokens. However, a sentence can contain several kinds of relationships simultaneously.

Consider:
> "The animal didn't cross the road because it was tired."

The model needs to learn that:
* `it` refers to `animal`.
* `tired` describes a condition.
* `cross` and `road` are related to the action and its context.

One attention head could learn to focus on some of these relationships. Multiple heads allow the model to learn different patterns in parallel.

Think of multiple heads as multiple learned ways of examining the same sequence, rather than multiple copies of the same attention calculation.

#### Multi-Head Attention Input token representations

![Multi-Head Attention Input token representations](image-29.png)

#### What does each attention head actually calculate?

Each head has its own learnable projection matrices:

$Q_h = XW_Q^{(h)}, \quad K_h = XW_K^{(h)}, \quad V_h = XW_V^{(h)}$

Each head then calculates attention:

$\text{Head}_h = \text{softmax}\left(\frac{Q_h K_h^\top}{\sqrt{d_k}}\right) V_h$

Notice that calculating the attention score is only part of the operation.

The process has three stages:
1. Calculate the similarity scores between queries and keys.
2. Apply softmax to convert the scores into attention weights.
3. Use those weights to combine the value vectors, producing contextual information for every token.

For example, the representation of `it` can incorporate information from `animal` if the learned attention weights give that token sufficient influence.

#### Why not use just one head?

Because one head has one learned set of query, key, and value projections. Multiple heads provide multiple independently learned attention patterns.

For the original Transformer architecture, the outputs from all heads are concatenated and projected:

$\text{MultiHead}(X) = \text{Concat}(\text{Head}_1, \dots, \text{Head}_H)W_O$

Here, $H$ is the number of attention heads, and $W_O$ is another learnable weight matrix.

For example, with $d_{\text{model}} = 768$ and 12 heads:

$d_k = d_v = \frac{768}{12} = 64$

Each head works with 64-dimensional query, key, and value representations. Their 12 outputs are concatenated to produce 768 features per token, before the final output projection preserves the model's hidden dimension.

### 3. What is the final output of Encoder ($N\times$)?

This is the most important part.

In the original Transformer diagram, **$N\times$ means the encoder layer is repeated N times**. Each layer contains:

1. Multi-head self-attention.
2. A residual connection and layer normalization (`Add & Norm`).
3. A feed-forward neural network.
4. Another residual connection and layer normalization.

The output of one encoder layer becomes the input to the next encoder layer.
The output is not merely an attention-score matrix. It is a sequence of contextualized token representations.

For a batch of sentences, the output shape is:

$(B, L, d_{\text{model}})$

where:
* $B$ = batch size.
* $L$ = padded sequence length.
* $d_{\text{model}}$ = hidden dimension.

![Input embeddings + positional information](image-30.png)

The final outcome is not just an attention-score matrix. It is a sequence of contextualized token vectors. Each token's representation incorporates information from other tokens through attention and subsequent neural-network layers.

## Decoder

### Now focus on Right part of the transformer architecture where the final outcome id Output probability from SoftMax. also called decoder. First tell me where the output (Shifter right) come from? Is it feed back from SoftMax out put. Why it is called decoder? From encoder architecture it's not much different. keep answer details. Do not sort answer.

Decoder looks similar to the encoder, but it has a different responsibility. Let's follow the original Transformer architecture from the input to the final Softmax output.

### Understanding the Right Side of the Transformer: The Decoder

Your questions are important because the decoder looks similar to the encoder, but it has a different responsibility. Let's follow the original Transformer architecture from the input to the final Softmax output.

#### 1. Where does the decoder's shifted-right input come from?

Yes, the shifted-right input is related to the target sequence, but it does not come directly from the Softmax output in the original training diagram.

In the original Transformer architecture, the decoder receives the target sentence shifted one position to the right **during training**. This is called *teacher forcing*.

Suppose we want the Transformer to translate:
* **Source sentence:** I love cats
* **Target sentence:** J'aime les chats

The target sequence is prepared with special start and end tokens:
> `[BOS] J'aime les chats [EOS]`

Here, `[BOS]` means beginning of sequence and `[EOS]` means end of sequence.

The decoder input is shifted right:
> `[BOS] J'aime les chats`

The expected prediction targets are:
> `J'aime les chats [EOS]`

The model learns to predict the next token at each position.

### Training example

**Decoder input (shifted right)**
> `[BOS]  J'aime  les  chats`

**Expected next-token targets**
> `[J'aime]  [les]  [chats]  [EOS]`

Each decoder position learns to predict the next token, using only the preceding target tokens and the available encoder information.

#### 2. Does the Softmax output feed back into the decoder?

The answer depends on whether the model is training or generating text.

##### During training

The shifted-right target sequence is supplied to the decoder. The decoder produces hidden representations, and a Linear layer followed by Softmax produces next-token probabilities.

For example, at a particular position:

| Candidate next token | Predicted probability |
| :--- | :--- |
| les | 0.85 |
| la | 0.10 |
| un | 0.05 |

The correct next token is compared with the prediction to calculate the loss. Backpropagation then updates the model's trainable weights.

The predicted token is not normally fed back as the next decoder input during standard teacher-forced training. Instead, the correct target sequence, shifted right, supplies the inputs for all positions.

##### During inference (generating a new sentence)

Now the model does not know the target sentence in advance. It must generate the next token itself.

For example, suppose the prompt is:
> `[BOS] I love`

The decoder predicts a probability distribution for the next token. Suppose the selected token is `cats`.

That token is appended to the existing sequence, and the decoder runs again to predict the next token.

![Current decoder input](image-31.png)

So, during autoregressive generation, the selected token from the model's output is fed back as part of the next decoder input. Technically, the model consumes the selected token's **token ID**, which is converted into an embedding; it does not feed the entire Softmax probability vector back into the decoder as its ordinary next input.

This is the principle behind text generation in GPT-style models.

#### 3. Why is it called a decoder?

In the original Transformer, the encoder processes the source sequence into contextual representations, and the decoder uses those representations along with previously generated target tokens to predict the target sequence.

For example, in translation:
* **Encoder:** Reads `I love cats` and creates contextual representations of the source tokens.
* **Decoder:** Uses the source representations and the target tokens generated so far to produce `J'aime les chats`.

The decoder is therefore responsible for generating the output sequence, one token at a time during inference.

The name comes from its role in transforming encoded source information and target-side context into the desired output sequence. In an encoder-decoder Transformer, the decoder does not simply reverse the encoder's calculations.

#### 4. If the decoder looks similar to the encoder, what is the actual difference?

Your observation is correct: both architectures contain Multi-Head Attention, Feed-Forward Networks, residual connections, and normalization. However, the decoder has additional behavior that makes it different.

Let's compare them.

![Encoder and decoder architecture](image-32.png)

The original Transformer decoder contains **two different attention mechanisms**:

1. **Masked self-attention:** Learns relationships among the target tokens generated so far.
2. **Cross-attention:** Allows the decoder to retrieve relevant information from the encoder's output.

The cross-attention mechanism is particularly important in the original translation architecture because it connects the source sentence with the target sentence being generated.

For example, when generating `chats`, the decoder can use both its existing target context, such as `J'aime les`, and relevant information from the encoded source sentence, `I love cats`.

##### Important distinction: GPT versus the original Transformer

Not every model called a Transformer has this same encoder–decoder architecture.

| Model | Architecture | How it generates or processes text |
| :--- | :--- | :--- |
| Original Transformer | Encoder + decoder | Decoder uses encoder outputs through cross-attention. |
| BERT / DistilBERT | Encoder-only | Produces contextual representations; not a standard autoregressive text generator. |
| GPT | Decoder-only | Uses masked self-attention to predict subsequent tokens; no separate encoder or encoder-decoder cross-attention. |

Therefore, when you study GPT, you will see masked self-attention and the output probability layer, but you will not see the separate encoder block shown in the original Transformer architecture.

#### 5. What exactly comes out of the decoder before Softmax?

The decoder does not immediately produce vocabulary probabilities. It first produces a hidden representation for each target position.

For example, suppose the decoder hidden size is 512. Its output might have shape:

$$H_{\text{decoder}} \in \mathbb{R}^{L \times 512}$$

A learned Linear layer maps these 512 features to a score for every vocabulary token:

$$Z = H_{\text{decoder}} W_{\text{vocab}} + b$$

If the vocabulary contains 30,000 tokens, the output at each position has 30,000 logits.

Softmax converts these logits into probabilities:

$$P(t_i \mid t_{<i}, X) = \text{softmax}(Z_i)$$

Here, $X$ represents the encoded source sequence in an encoder–decoder model, and $t_{<i}$ represents the preceding target tokens.

The probabilities indicate how likely each vocabulary token is to be the next token, given the available context.

**Final understanding:** The decoder receives shifted-right target tokens during training and previously generated tokens during inference. It uses masked self-attention to understand the target-side context and, in the original encoder–decoder Transformer, cross-attention to access encoder information. Its final hidden representations are converted by Linear and Softmax layers into next-token probabilities. During generation, the selected token is appended to the input, and the process repeats until the sequence ends.

